# generación de manifests

Este bloque regenera los artefactos derivados a partir de `master_manifest.csv` y `recordings_manifest.csv`.

Decisiones fijadas:

- Las especies objetivo \(K\) deben:
  - tener al menos **200 eventos válidos para clasificación**
  - aparecer en al menos **15 grabaciones**
  - tener evidencia en un dataset colombiano: `PteroSet` o `WABAD-CO`
- El detector utiliza únicamente eventos con `use_detector == True`
- El clasificador utiliza únicamente eventos con `use_classifier == True`
- `OTHER_SPECIES` contiene taxones a nivel de especie que:
  - tienen evidencia en `PteroSet` o `WABAD-CO`
  - no pertenecen a \(K\)

In [14]:
from pathlib import Path
import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:

# rutas del proyecto
BASE_DIR = Path("/content/drive/MyDrive/DATASET_PROYECTO")
METADATA_DIR = BASE_DIR / "metadata"
STATISTICS_DIR = BASE_DIR / "statistics"
MASTER_PATH = METADATA_DIR / "master_manifest.csv"
RECORDINGS_PATH = METADATA_DIR / "recordings_manifest.csv"
TARGET_PATH = METADATA_DIR / "target_species.csv"
DETECTOR_PATH = METADATA_DIR / "detector_manifest.csv"
CLASSIFIER_PATH = METADATA_DIR / "classifier_manifest.csv"
SUMMARY_PATH = STATISTICS_DIR / "dataset_summary.csv"

STATISTICS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MASTER_PATH:", MASTER_PATH)
print("RECORDINGS_PATH:", RECORDINGS_PATH)


BASE_DIR: /content/drive/MyDrive/DATASET_PROYECTO
MASTER_PATH: /content/drive/MyDrive/DATASET_PROYECTO/metadata/master_manifest.csv
RECORDINGS_PATH: /content/drive/MyDrive/DATASET_PROYECTO/metadata/recordings_manifest.csv



## 1. Cargar y validar los manifests base

`master_manifest.csv` es la fuente de verdad de los eventos

`recordings_manifest.csv` es el csv de grabaciones físicas


In [16]:

master = pd.read_csv(MASTER_PATH)
recordings_manifest = pd.read_csv(RECORDINGS_PATH)

required_master = {
    "event_id", "dataset", "country", "species_scientific",
    "t_min", "t_max", "f_min", "f_max",
    "use_detector", "use_classifier", "recording_uid"
}
required_recordings = {
    "recording_uid", "dataset", "duration_sec"
}

missing_master = required_master - set(master.columns)
missing_recordings = required_recordings - set(recordings_manifest.columns)

assert not missing_master, f"Faltan columnas en master_manifest: {missing_master}"
assert not missing_recordings, f"Faltan columnas en recordings_manifest: {missing_recordings}"

assert master["event_id"].is_unique, "event_id debe ser único en master_manifest."
assert recordings_manifest["recording_uid"].is_unique, \
    "recording_uid debe ser único en recordings_manifest."

missing_recordings_in_inventory = (
    set(master["recording_uid"].dropna())
    - set(recordings_manifest["recording_uid"].dropna())
)
assert not missing_recordings_in_inventory, (
    f"Hay recording_uid del master que no existen en recordings_manifest: "
    f"{len(missing_recordings_in_inventory)}"
)

print("Eventos master:", len(master))
print("Grabaciones inventariadas:", len(recordings_manifest))
print("Grabaciones únicas:", recordings_manifest["recording_uid"].nunique())


Eventos master: 48465
Grabaciones inventariadas: 1011
Grabaciones únicas: 1011



## 2. Normalización taxonómica

No se reescribe `master_manifest.csv`. La normalización solo asegura que las comparaciones de especies sean reproducibles

Se conservan los `NaN`: no se convierten al texto `"nan"`.


In [17]:
TAXON_FIXES = {
    "melanerpes cruentats": "melanerpes cruentatus",
    "megarynchus pitagua": "megarynchus pitangua",
    "mesembrinibis cayanensis": "mesembrinibis cayennensis",
}

master["species_scientific"] = (master["species_scientific"].astype("string").str.normalize("NFKC").str.strip().str.casefold().str.replace(r"\s+", " ", regex=True).replace(TAXON_FIXES))

# etiqueta taxonómica a nivel de especie:
# acepta binomios y, por robustez, trinomios; excluye etiquetas como picidae, psittacidae sp., tyrannidae sp 1 o caracteres corruptos
SPECIES_LEVEL_PATTERN = (
    r"^[a-z]+(?:-[a-z]+)? "
    r"[a-z]+(?:-[a-z]+)?"
    r"(?: [a-z]+(?:-[a-z]+)?)?$"
)

species_level_mask = (master["species_scientific"].str.match(SPECIES_LEVEL_PATTERN, na=False))

excluded_taxon_labels = sorted(master.loc[master["species_scientific"].notna() & ~species_level_mask,"species_scientific"].unique())

print("Etiquetas no consideradas especies cientificas completas:")
print(excluded_taxon_labels)


Etiquetas no consideradas especies cientificas completas:
['picidae', 'psittacidae sp.', 'tyrannidae sp 1', '–']



## 3. Regenerar `target_species.csv`

El soporte de una especie se calcula únicamente sobre eventos utilizables por el clasificador (`use_classifier == True`).

Una especie puede recibir ejemplos adicionales de Col-CR o Perú, pero **solo puede entrar en \(K\)** si tiene al menos una aparición válida en `PteroSet` o `WABAD-CO`.


In [18]:
MIN_EVENTS = 200
MIN_RECORDINGS = 15
CO_DATASETS = {"PteroSet", "WABAD-CO"}

classifier_species_base = master.loc[master["use_classifier"].fillna(False) & master["species_scientific"].notna() & species_level_mask].copy()

species_colombia = set(classifier_species_base.loc[classifier_species_base["dataset"].isin(CO_DATASETS),"species_scientific"].unique())

species_stats = (classifier_species_base.groupby("species_scientific").agg(eventos=("event_id", "count"), grabaciones=("recording_uid", "nunique") ,datasets=("dataset", "nunique"),))

K_candidates = species_stats.loc[species_stats.index.isin(species_colombia) & (species_stats["eventos"] >= MIN_EVENTS) & (species_stats["grabaciones"] >= MIN_RECORDINGS)].copy()

target_species = (K_candidates.reset_index()[["species_scientific"]].sort_values("species_scientific").drop_duplicates().reset_index(drop=True))

target_species.insert(0, "code", range(1, len(target_species) + 1))
target_species["species_norm"] = target_species["species_scientific"]

K_SPECIES = set(target_species["species_scientific"])

assert target_species["species_scientific"].notna().all()
assert target_species["species_scientific"].is_unique
assert "nan" not in K_SPECIES

print("Numero de especies objetivo K:", len(target_species))
display(K_candidates.loc[target_species["species_scientific"]].reset_index())


Numero de especies objetivo K: 30


,species_scientific,eventos,grabaciones,datasets
0,arremon brunneinucha,717,62,2
1,atlapetes schistaceus,252,29,1
2,attila spadiceus,200,52,3
3,camptostoma obsoletum,255,19,2
4,campylorhynchus turdinus,951,29,2
5,cantorchilus leucotis,824,44,3
6,colibri cyanotus,498,51,1
7,crypturellus cinereus,789,173,2
8,crypturellus soui,248,28,3
9,crypturellus undulatus,611,60,2



## 4. Regenerar `detector_manifest.csv`

Para detección se conservan todos los eventos válidos, tengan o no identificación taxonómica.

La condición es exclusivamente:


use_detector == True

Esto elimina anotaciones inválidas sin descartar los eventos no identificados que si son útiles


In [19]:
detector_manifest = master.loc[master["use_detector"].fillna(False)].copy()

# evitamos propagar etiquetas antiguas calculadas con otro K
detector_manifest = detector_manifest.drop(columns=["is_target", "class_label"],errors="ignore")
detector_manifest["detector_label"] = 1

# Validaciones
assert (detector_manifest["t_max"] > detector_manifest["t_min"]).all()
assert (detector_manifest["f_max"] > detector_manifest["f_min"]).all()
assert (detector_manifest["t_min"] >= 0).all()
assert (detector_manifest["f_min"] >= 0).all()
assert detector_manifest["event_id"].is_unique

print("Eventos detector:", len(detector_manifest))
print("Grabaciones detector:", detector_manifest["recording_uid"].nunique())
print("Eventos sin identificacion taxonomica conservados:",detector_manifest["species_scientific"].isna().sum())


Eventos detector: 48430
Grabaciones detector: 1010
Eventos sin identificacion taxonomica conservados: 10739



## 5. Regenerar `classifier_manifest.csv`

El manifest del clasificador representa **todos los eventos elegibles**, no una muestra balanceada.

- `class_label`: etiqueta supervisada actual.
- `class_code = 0`: `OTHER_SPECIES`.
- `class_code = 1..K`: especies objetivo, siguiendo `target_species.csv`.
- `species_scientific` conserva la taxonomía original.
- `original_species` se conserva explícitamente para auditoría.

El balanceo de `OTHER_SPECIES` se hará después del split, únicamente sobre `train`. Así `validation` y `test` mantienen la distribución observada y las métricas no quedan artificialmente alteradas.


In [20]:
classifier_base = master.loc[master["use_classifier"].fillna(False) & master["species_scientific"].notna() & species_level_mask].copy()

classifier_target = classifier_base.loc[classifier_base["species_scientific"].isin(K_SPECIES)].copy()

classifier_other = classifier_base.loc[classifier_base["species_scientific"].isin(species_colombia) & ~classifier_base["species_scientific"].isin(K_SPECIES)].copy()

# recalcular etiquetas derivadas, no usar los valores antiguos del master
classifier_target["is_target"] = True
classifier_target["class_label"] = classifier_target["species_scientific"]
classifier_target["original_species"] = classifier_target["species_scientific"]
classifier_other["is_target"] = False
classifier_other["class_label"] = "OTHER_SPECIES"
classifier_other["original_species"] = classifier_other["species_scientific"]

code_map = dict(zip(target_species["species_scientific"],target_species["code"]))
classifier_target["class_code"] = (classifier_target["species_scientific"].map(code_map).astype("int64"))
classifier_other["class_code"] = 0

classifier_manifest = pd.concat([classifier_target, classifier_other],ignore_index=True)

classifier_manifest = (classifier_manifest.sort_values(["dataset", "recording_uid", "t_min", "event_id"]).reset_index(drop=True))

assert classifier_manifest["event_id"].is_unique
assert classifier_manifest["class_label"].notna().all()
assert classifier_manifest["class_code"].notna().all()
assert set(classifier_manifest["class_label"].unique()) == (K_SPECIES | {"OTHER_SPECIES"})

print("Eventos classifier:", len(classifier_manifest))
print("Grabaciones classifier:", classifier_manifest["recording_uid"].nunique())
print("Clases supervisadas:", classifier_manifest["class_label"].nunique())
print("Eventos target:", int(classifier_manifest["is_target"].sum()))
print("Eventos OTHER_SPECIES:", int((classifier_manifest["class_label"] == "OTHER_SPECIES").sum()))


Eventos classifier: 25589
Grabaciones classifier: 818
Clases supervisadas: 31
Eventos target: 14405
Eventos OTHER_SPECIES: 11184



## 6. Auditoría específica de `OTHER_SPECIES`

Esta validación comprueba que ninguna especie exclusiva de Perú o Costa Rica haya entrado en `OTHER_SPECIES`


In [21]:
other_original_species = set(classifier_manifest.loc[classifier_manifest["class_label"].eq("OTHER_SPECIES"),"original_species"].unique())

outside_colombia_evidence = other_original_species - species_colombia
target_leakage_into_other = other_original_species & K_SPECIES

assert not outside_colombia_evidence, ("Hay especies en OTHER_SPECIES sin evidencia en PteroSet/WABAD-CO: "f"{sorted(outside_colombia_evidence)}")
assert not target_leakage_into_other, ("Hay especies K dentro de OTHER_SPECIES: "f"{sorted(target_leakage_into_other)}")

print("Especies originales agrupadas en OTHER_SPECIES:", len(other_original_species))

Especies originales agrupadas en OTHER_SPECIES: 261



## 7. Regenerar `dataset_summary.csv`

El resumen se deriva de los manifests finales


In [22]:
recording_summary = (recordings_manifest.groupby("dataset").agg(recordings_selected=("recording_uid", "nunique"),total_duration_sec=("duration_sec", "sum"),))

recording_summary["hours_available"] = (recording_summary["total_duration_sec"] / 3600.0)

detector_counts = detector_manifest.groupby("dataset").size()
classifier_counts = classifier_manifest.groupby("dataset").size()
target_counts = (classifier_manifest.loc[classifier_manifest["is_target"]].groupby("dataset").size())
other_counts = (classifier_manifest.loc[classifier_manifest["class_label"].eq("OTHER_SPECIES")].groupby("dataset").size())

dataset_summary = (recording_summary[["recordings_selected", "hours_available"]].assign(detector_events=detector_counts, classifier_events=classifier_counts, target_events=target_counts, other_species_events=other_counts,).fillna(0).reset_index())

for col in ["recordings_selected","detector_events","classifier_events","target_events","other_species_events",]:
    dataset_summary[col] = dataset_summary[col].astype(int)

assert (dataset_summary["classifier_events"] == dataset_summary["target_events"] + dataset_summary["other_species_events"]).all()

display(dataset_summary)

,dataset,recordings_selected,hours_available,detector_events,classifier_events,target_events,other_species_events
0,Col-CR,34,33.997249,7338,3117,1779,1338
1,Peru,21,21.000000,16480,6583,5180,1403
2,PteroSet,563,73.620556,15371,6648,3284,3364
3,WABAD-CO,393,6.549966,9241,9241,4162,5079



## 8. Validaciones de esta versión del corpus

Estas cifras son invariantes esperadas para los archivos base actuales. Si alguna falla, **no se debe sobrescribir silenciosamente**: hay que revisar qué cambió en `master_manifest.csv` o `recordings_manifest.csv`.



## 9. Guardar manifests corregidos

Esta celda  sobreescribe

- `target_species.csv`
- `detector_manifest.csv`
- `classifier_manifest.csv`
- `dataset_summary.csv`


In [23]:
def save_csv_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)

save_csv_atomic(target_species, TARGET_PATH)
save_csv_atomic(detector_manifest, DETECTOR_PATH)
save_csv_atomic(classifier_manifest, CLASSIFIER_PATH)
save_csv_atomic(dataset_summary, SUMMARY_PATH)

print("Guardados:")
print(" -", TARGET_PATH)
print(" -", DETECTOR_PATH)
print(" -", CLASSIFIER_PATH)
print(" -", SUMMARY_PATH)


Guardados:
 - /content/drive/MyDrive/DATASET_PROYECTO/metadata/target_species.csv
 - /content/drive/MyDrive/DATASET_PROYECTO/metadata/detector_manifest.csv
 - /content/drive/MyDrive/DATASET_PROYECTO/metadata/classifier_manifest.csv
 - /content/drive/MyDrive/DATASET_PROYECTO/statistics/dataset_summary.csv
